# Local 01: Dataset Upload to Hugging Face

로컬 환경(Jupyter/VSCode Notebook)에서 현재 프로젝트 데이터셋을 Hugging Face dataset repo로 업로드합니다.


In [1]:
!pip -q install -U huggingface_hub hf_transfer


## 경로/토큰/업로드 대상 설정


In [ ]:
import os
from pathlib import Path
from huggingface_hub import login

# 로컬 프로젝트 루트 자동 인식 (Windows / WSL)
CANDIDATE_ROOTS = [
    r'C:\\Users\\User\\Desktop\\LLM_JSSP_masking',
    '/mnt/c/Users/User/Desktop/LLM_JSSP_masking',
]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if Path(p).exists()), CANDIDATE_ROOTS[-1])
PROJECT_ROOT = str(Path(PROJECT_ROOT).expanduser().resolve())
root = Path(PROJECT_ROOT)
print('PROJECT_ROOT:', root)
assert root.exists(), f'프로젝트 경로 없음: {root}'

######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''

if not HF_TOKEN:
    print("HF_TOKEN is empty. Fill the block above or set a Colab secret/env var before Hugging Face operations.")
else:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login ready')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

UPLOAD_RAW_SOURCE = False
UPLOAD_STEP_DATASETS = True
UPLOAD_VALIDATION_DATA = False

DATASET_SPECS = {
    'HYUNJINI/jssp_raw_source_v1': [
        root / 'llm_jssp' / 'train.json',
    ],
    'HYUNJINI/jssp_policy_step_train_all_v1': [
        root / 'train_data' / 'jssp_step_train_policy.jsonl',
        root / 'train_data' / 'jssp_step_train_policy.summary.json',
    ],
    'HYUNJINI/jssp_reason_step_train_all_v1': [
        root / 'train_data' / 'jssp_step_train_reason.jsonl',
        root / 'train_data' / 'jssp_step_train_reason.summary.json',
    ],
    'HYUNJINI/jssp_mixed_step_train_all_v1': [
        root / 'train_data' / 'jssp_step_train_with_reason.jsonl',
        root / 'train_data' / 'jssp_step_train_with_reason.summary.json',
    ],
    'HYUNJINI/jssp_policy_step_train_dispatch_v1': [
        root / 'train_data' / 'jssp_step_train_policy_dispatch.jsonl',
        root / 'train_data' / 'jssp_step_train_policy_dispatch.summary.json',
    ],
    'HYUNJINI/jssp_reason_step_train_dispatch_v1': [
        root / 'train_data' / 'jssp_step_train_reason_dispatch.jsonl',
        root / 'train_data' / 'jssp_step_train_reason_dispatch.summary.json',
    ],
    'HYUNJINI/jssp_mixed_step_train_dispatch_v1': [
        root / 'train_data' / 'jssp_step_train_with_reason_dispatch.jsonl',
        root / 'train_data' / 'jssp_step_train_with_reason_dispatch.summary.json',
    ],
    'HYUNJINI/jssp_validation_all_v1': [
        root / 'validation_data' / 'custom_step_test.json',
        root / 'validation_data' / 'la.json',
        root / 'validation_data' / 'ta.json',
    ],
}

ENABLED_REPOS = {}
for repo_id, file_paths in DATASET_SPECS.items():
    if repo_id == 'HYUNJINI/jssp_raw_source_v1' and not UPLOAD_RAW_SOURCE:
        continue
    if repo_id == 'HYUNJINI/jssp_validation_all_v1' and not UPLOAD_VALIDATION_DATA:
        continue
    if repo_id not in {'HYUNJINI/jssp_raw_source_v1', 'HYUNJINI/jssp_validation_all_v1'} and not UPLOAD_STEP_DATASETS:
        continue

    existing = [p for p in file_paths if p.exists()]
    if existing:
        ENABLED_REPOS[repo_id] = existing

print()
print('=== Upload Plan ===')
if not ENABLED_REPOS:
    print('업로드할 로컬 파일이 없습니다.')
else:
    for repo_id, files in ENABLED_REPOS.items():
        print()
        print(f'[{repo_id}] {len(files)} files')
        for fp in files:
            size_gb = fp.stat().st_size / (1024 ** 3)
            print(f' - {fp.relative_to(root)} ({size_gb:.3f} GB)')


PROJECT_ROOT: C:\Users\User\Desktop\LLM_JSSP_masking
HF login ready

=== Upload Plan ===

[HYUNJINI/jssp_mixed_step_train_all_v1] 2 files
 - train_data\jssp_step_train_with_reason.jsonl (77.984 GB)
 - train_data\jssp_step_train_with_reason.summary.json (0.000 GB)

[HYUNJINI/jssp_policy_step_train_dispatch_v1] 2 files
 - train_data\jssp_step_train_policy_dispatch.jsonl (16.145 GB)
 - train_data\jssp_step_train_policy_dispatch.summary.json (0.000 GB)

[HYUNJINI/jssp_mixed_step_train_dispatch_v1] 2 files
 - train_data\jssp_step_train_with_reason_dispatch.jsonl (44.046 GB)
 - train_data\jssp_step_train_with_reason_dispatch.summary.json (0.000 GB)


## 업로드 실행


In [ ]:
from huggingface_hub import HfApi, login

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is empty. Fill the visible HF_TOKEN block in the setup cell before uploading.")


api = HfApi(token=HF_TOKEN)

if not ENABLED_REPOS:
    print('업로드할 파일이 없어서 종료합니다.')
else:
    for repo_id, file_paths in ENABLED_REPOS.items():
        print(f'\n=== Repo: {repo_id} ===')
        api.create_repo(repo_id=repo_id, repo_type='dataset', private=False, exist_ok=True)

        for local_path in file_paths:
            if not local_path.exists():
                print(f'SKIP missing: {local_path}')
                continue
            size_gb = local_path.stat().st_size / (1024 ** 3)
            rel = local_path.relative_to(root)
            print(f'Uploading {rel} ({size_gb:.2f} GB)')
            api.upload_file(
                path_or_fileobj=str(local_path),
                path_in_repo=str(rel).replace('\\', '/'),
                repo_id=repo_id,
                repo_type='dataset',
            )

    print('\n선택된 dataset 업로드 완료')



=== Repo: HYUNJINI/jssp_mixed_step_train_all_v1 ===
Uploading train_data\jssp_step_train_with_reason.jsonl (77.98 GB)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

## 업로드 검증


In [ ]:
for repo_id in ENABLED_REPOS.keys():
    files = api.list_repo_files(repo_id=repo_id, repo_type='dataset')
    print(f'\n{repo_id} ({len(files)} files)')
    for f in files:
        print(' -', f)
